# VocEd Lab 6 — Random Forest, MLP, and the Colour Ceiling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/Labs/Lab6_random_forest_mlp_colour_ceiling.ipynb)

Lab 5 gave each pixel its three colour values and let k-NN and logistic regression learn the
rule.  They ended up about level with Lab 4's thresholds.  Was that because the classifiers were
too simple?  This lab tries two much more flexible ones, and then asks a sharper question:

- a **random forest** — hundreds of yes/no questions about the colour, voted together;
- a **multi-layer perceptron (MLP)** — a small neural network;
- and the **colour ceiling**: the best score that *any* rule which looks at one pixel's colour
  could possibly reach.  We can measure it directly.

**Textbook:** [Chapter 5](https://obooks.tech/imageprocessing/chapter5.html) (Methods 3 and 4, and
*Why All Colour-Only Methods Hit a Ceiling*).

---

### How the puzzles work 🧩🤔

- **🧩 Code puzzles:** replace every `___` and run the cell — a ✅ means you got it.
- **🤔 Think puzzles:** answer in the ✏️ cell below the question.  There is no check: discuss
  your answer with your group.

## 0. Setup — Get the Data

The same images and split as Labs 3–5, and Lab 5's **random** training table: 450 pixels from
anywhere in each of the 120 training images.

In [ ]:
!git clone https://github.com/emilsar/VocEd.git
%cd VocEd

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

X = np.stack([np.load(f"imagedata/X/{i}.npy") for i in range(200)])   # (200, 3, 256, 256)
y = np.stack([np.load(f"imagedata/y/{i}.npy") for i in range(200)])   # (200, 256, 256)

nuc_px   = (y == 2).sum(axis=(1, 2))
quartile = np.digitize(nuc_px, np.percentile(nuc_px, [25, 50, 75]))
trainval_idx, test_idx = train_test_split(np.arange(200), test_size=0.2, stratify=quartile, random_state=42)
train_idx, val_idx = train_test_split(trainval_idx, test_size=40, stratify=quartile[trainval_idx], random_state=42)
assert list(test_idx[:5]) == [121, 152, 81, 4, 141], "Not the same test images as Labs 2–5"

def dice_score(pred, target, label):
    pred_mask   = (pred   == label)
    target_mask = (target == label)
    denominator = pred_mask.sum() + target_mask.sum()
    if denominator == 0:
        return 1.0
    return 2 * (pred_mask & target_mask).sum() / denominator

def image_dice(pred, mask):
    return (dice_score(pred, mask, 1) + dice_score(pred, mask, 2)) / 2      # cytoplasm and nucleus

def nc_ratio(mask):
    return (mask == 2).sum() / max((mask == 1).sum(), 1)

def pixels_of(i):
    return X[i].transpose(1, 2, 0).reshape(-1, 3)                           # (65536, 3)

def predict_mask(classifier, i):
    return classifier.predict(pixels_of(i)).reshape(256, 256)

def val_dice(classifier):
    return np.mean([image_dice(predict_mask(classifier, i), y[i]) for i in val_idx])

# Lab 5's random training table
rng = np.random.default_rng(0)
rows, row_labels = [], []
for i in train_idx:
    chosen = rng.choice(65536, 450, replace=False)
    rows.append(pixels_of(i)[chosen])
    row_labels.append(y[i].reshape(-1)[chosen])
X_px, y_px = np.vstack(rows), np.concatenate(row_labels)
print("Training table:", X_px.shape)

**Scores so far, on the 40 validation images:** Lab 4 thresholds + clean-up 0.812;
Lab 5 logistic regression about 0.80.

## 1. Random Forest

A **decision tree** labels a pixel by asking a chain of yes/no questions — *is green below
0.45?  then is blue above 0.38?* — and the answers lead to a leaf with a label.  A **random
forest** grows 100 trees, each on a random part of the table, and lets them vote.

`min_samples_leaf` sets how many training pixels every leaf must hold.  With 1, a tree can keep
asking until each leaf holds a single pixel.  *(About a minute for the three forests.)*

📖 Book: [§5.7 Method 3 — Random Forest](https://obooks.tech/imageprocessing/chapter5.html#method-3-random-forest)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forests = {}
for leaf in [1, 20, 100]:
    rf = RandomForestClassifier(n_estimators=100, min_samples_leaf=leaf, random_state=0).fit(X_px, y_px)
    forests[leaf] = rf
    print(f"min_samples_leaf = {leaf:3d}   training accuracy {rf.score(X_px, y_px):.3f}   validation Dice {val_dice(rf):.4f}")

### 🤔 Think 1 — Leaves that hold one pixel

1. Which forest has the best training accuracy, and which the best validation Dice?
2. With `min_samples_leaf = 1`, what does a leaf in the cytoplasm–nucleus overlap end up
   doing?  Which Lab 5 model does that remind you of?
3. `min_samples_leaf` is a knob like `k` in k-NN.  On which images should we choose it?

📖 Book: [Appendix C — Overfitting and Underfitting](https://obooks.tech/imageprocessing/appendixc.html#overfitting-and-underfitting)

✏️ **Your answer:** *(double-click this cell and type here)*

### 🧩 Puzzle 1 — What did the forest look at?

A forest records how much each feature helped its questions: `feature_importances_`, one number
per column, adding up to 1.  Which colour mattered most to the `min_samples_leaf = 100` forest?

📖 Book: [§5.7 Method 3 — Random Forest](https://obooks.tech/imageprocessing/chapter5.html#method-3-random-forest)

<details><summary><b>💡 Hint</b> (click to open)</summary>

You want the position of the largest importance.

</details>

In [ ]:
rf = forests[100]
colours = ['red', 'green', 'blue']
print("Importances:", dict(zip(colours, rf.feature_importances_.round(3))))

most_used = colours[np.___(rf.feature_importances_)]
print("Most important colour:", most_used)

# ── Check ──
assert most_used == colours[int(np.argmax(rf.feature_importances_))], "Take the largest importance"
print("✅ Puzzle 1 solved!")

The same answer as logistic regression's weights in Lab 5: **green**.  Two very different models
agree on what the data says.

## 2. The Multi-Layer Perceptron

An MLP passes the three colours through layers of **neurons**.  Each neuron is a logistic
regression of the layer before, followed by a bend (ReLU) that lets the network draw curved
boundaries.  Ours has 32 neurons, then 16, then 3 outputs: 3 → 32 → 16 → 3.  *(About 15 seconds.)*

### 🧩 Puzzle 2 — Count the weights

Between two layers, every neuron is connected to every neuron of the next layer, and each neuron
also has one constant (bias).  Fill in the middle connection.

📖 Book: [§5.8 Method 4 — Multi-Layer Perceptron](https://obooks.tech/imageprocessing/chapter5.html#method-4-multi-layer-perceptron)

<details><summary><b>💡 Hint</b> (click to open)</summary>

3 inputs to 32 neurons is 3 × 32 weights.  What connects 32 neurons to 16?

</details>

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=200, random_state=0).fit(X_px, y_px)
print(f"MLP validation Dice: {val_dice(mlp):.4f}")

n_weights = 3 * 32 + ___ + 16 * 3
n_biases  = 32 + 16 + 3
print(f"Weights {n_weights} + biases {n_biases} = {n_weights + n_biases} numbers to learn")
print("Logistic regression in Lab 5 learned 3 × 3 weights + 3 constants = 12")

# ── Check ──
assert n_weights == sum(W.size for W in mlp.coefs_), "One weight for every pair of connected neurons"
print("✅ Puzzle 2 solved!")

### 🤔 Think 2 — 60 times the weights

The MLP learns about 60 times as many numbers as logistic regression.

1. How much better is its validation Dice than logistic regression's (about 0.80) and the best
   forest's?
2. The MLP and the forest can both draw curved, complicated boundaries.  Logistic regression only
   draws flat planes.  If complicated boundaries were what the data needed, what would you expect
   to see in the scores?  What do you see?

📖 Book: [§5.8 Method 4 — Multi-Layer Perceptron](https://obooks.tech/imageprocessing/chapter5.html#method-4-multi-layer-perceptron)

✏️ **Your answer:** *(double-click this cell and type here)*

## 3. The Colour Ceiling

Any rule that looks only at a pixel's colour must give **every pixel of the same colour the same
label**.  So the best such a rule can do is: for each colour, pick the label that pixels of that
colour have most often.  We can compute that best possible rule directly.

1. Round every colour to one of 32 levels per channel, so similar colours share a **bin**
   (32 × 32 × 32 = 32,768 bins).
2. Count, in each bin, how many pixels are background, cytoplasm and nucleus.
3. The best rule labels each bin with its most common class.

We do it on the **validation images themselves** — fitted on the very pixels it is scored on.
That is cheating in the rule's favour, which is the point: no colour-only classifier trained on
other images can beat its **pixel accuracy** (the share of pixels labelled right) — picking the most
common label for every colour is exactly what maximises it.

### 🧩 Puzzle 3 — The best label for each colour

`counts[b]` holds the three class counts of bin `b`.  The best label is the class with the
largest count — along which axis?

📖 Book: [§5.10 Why All Colour-Only Methods Hit a Ceiling](https://obooks.tech/imageprocessing/chapter5.html#why-all-colour-only-methods-hit-a-ceiling)

<details><summary><b>💡 Hint</b> (click to open)</summary>

`counts` has shape (32768, 3): one row per bin, one column per class.  Axis 0 runs down the rows, axis 1 along each row.

</details>

In [ ]:
def colour_bin(pixels):
    levels = (np.minimum(pixels, 0.9999) * 32).astype(int)                  # 0 … 31 for each channel
    return levels[:, 0] * 32 * 32 + levels[:, 1] * 32 + levels[:, 2]        # one number per bin

bins   = np.concatenate([colour_bin(pixels_of(i)) for i in val_idx])
labels = np.concatenate([y[i].reshape(-1) for i in val_idx])

counts = np.zeros((32768, 3), dtype=int)
np.add.at(counts, (bins, labels), 1)                                        # counts[bin, label] += 1

best_label = counts.argmax(axis=___)
ceiling_masks = best_label[bins].reshape(40, 256, 256)

ceiling_dice = np.mean([image_dice(m, y[i]) for m, i in zip(ceiling_masks, val_idx)])
ceiling_acc  = np.mean(best_label[bins] == labels)
mixed = (counts > 0).sum(axis=1) > 1
print(f"Pixels whose colour bin also holds another class: {counts[mixed].sum() / counts.sum():.1%}")
print(f"Colour ceiling: pixel accuracy {ceiling_acc:.4f}   validation Dice {ceiling_dice:.4f}")

for name, model in [('random forest (leaf 100)', forests[100]), ('MLP', mlp)]:
    acc = np.mean(np.concatenate([model.predict(pixels_of(i)) for i in val_idx]) == labels)
    print(f"  {name:26s} pixel accuracy {acc:.4f}   validation Dice {val_dice(model):.4f}")

# ── Check ──
assert best_label.shape == (32768,), "One best label per bin: the largest count in each row"
print("✅ Puzzle 3 solved!")

Where does even the best colour-only rule go wrong?  Here is the first validation image:

In [ ]:
i0 = val_idx[0]
wrong = ceiling_masks[0] != y[i0]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(X[i0].transpose(1, 2, 0))
axes[0].set_title(f'validation image {i0}')
axes[1].imshow(y[i0], cmap='viridis', vmin=0, vmax=2)
axes[1].set_title('true mask')
axes[2].imshow(ceiling_masks[0], cmap='viridis', vmin=0, vmax=2)
axes[2].set_title('best colour-only rule')
axes[3].imshow(wrong, cmap='gray')
axes[3].set_title(f'wrong pixels (white): {wrong.mean():.1%}')
for ax in axes:
    ax.axis('off')
plt.show()

### 🤔 Think 3 — Reading the ceiling

1. What share of pixels have a colour that also occurs in another class?  What does that say
   about "finding the right colour rule"?
2. Compare the ceiling's pixel accuracy with the forest's and the MLP's.  How much room was left
   for a better colour-only classifier?  (Their Dice is even a little *above* the ceiling's — the
   ceiling maximises accuracy, not Dice — so compare accuracies.)
3. Look at the wrong pixels.  Are they scattered evenly, or concentrated somewhere?  What
   information — not in the pixel's own colour — would fix them?
4. Why do we need *bins* at all?  What would happen to this "ceiling" with a million bins, one
   for nearly every pixel?

📖 Book: [§5.10 Why All Colour-Only Methods Hit a Ceiling](https://obooks.tech/imageprocessing/chapter5.html#why-all-colour-only-methods-hit-a-ceiling)

✏️ **Your answer:** *(double-click this cell and type here)*

## 4. The Final Test — Once

The forest (`min_samples_leaf = 100`) and the MLP, on the 40 test images.

📖 Book: [§5.9 Comparing the Methods](https://obooks.tech/imageprocessing/chapter5.html#comparing-the-methods)

In [ ]:
nc_true = np.array([nc_ratio(y[i]) for i in test_idx])

print(f"{'':28s} test Dice   N/C R²   N/C within ±0.05")
for name, model in [('random forest (leaf 100)', forests[100]), ('MLP (3 → 32 → 16 → 3)', mlp)]:
    preds   = [predict_mask(model, i) for i in test_idx]
    dice    = np.mean([image_dice(p, y[i]) for p, i in zip(preds, test_idx)])
    nc_pred = np.array([nc_ratio(p) for p in preds])
    r2      = 1 - np.sum((nc_pred - nc_true) ** 2) / np.sum((nc_true - nc_true.mean()) ** 2)
    close   = np.mean(np.abs(nc_pred - nc_true) < 0.05)
    print(f"{name:28s} {dice:.4f}    {r2:7.3f}   {close:.0%}")

print()
print("Earlier labs, on the same 40 test images:")
print("  Lab 3 five-knob pipeline        0.770     -0.812")
print("  Lab 4 specks, fill              0.772     -0.465   35%")

### 🤔 Think 4 — Two roads, one wall

1. How do the forest and the MLP compare with Lab 4 on the test images?
2. Labs 2–4 used *one* brightness value plus hand-written clean-up.  Labs 5–6 used *three* colour
   values and learned the rule.  Both roads arrive at about the same Dice.  Using the ceiling,
   explain why more colour alone could not go much further.
3. Lab 4's clean-up steps looked at *blobs* — groups of neighbouring pixels.  In that sense, Lab 4
   already used a little neighbourhood information.  How does that help explain why thresholds +
   clean-up keep up with the flexible colour classifiers?

📖 Book: [§5.10 Why All Colour-Only Methods Hit a Ceiling](https://obooks.tech/imageprocessing/chapter5.html#why-all-colour-only-methods-hit-a-ceiling)

✏️ **Your answer:** *(double-click this cell and type here)*

## Wrap-up

**Key takeaways:**
- **Random forests and MLPs can draw any boundary** — and both, like k-NN with `k = 1`, can memorise.
  Leaf size (and the number of neighbours, and how long you train) are knobs for that; choose them on
  validation.
- **Different models agreeing is a clue.**  Logistic regression, forest and MLP all say green matters
  most; forest and MLP reach the same Dice.  When very different models hit the same score, look at
  the input.
- **The colour ceiling is measurable.**  For each colour, the best possible label is the most common
  one.  About 99% of pixels share their colour with another class, and our classifiers already sit at
  the ceiling.
- **A pixel's colour is not enough.**  The remaining errors are pale patches inside nuclei and dark
  grains in the cytoplasm — mistakes that only the pixel's neighbourhood can explain.  Lab 7 gives the
  classifier that neighbourhood.

---

### Group discussion 👥

Split the work, then share your numbers:

| Person | Change, then re-run |
|---|---|
| A | Section 1 with `n_estimators=20` instead of 100 |
| B | Section 2 with `hidden_layer_sizes=(8,)` — one layer of 8 neurons |
| C | Section 3 with 16 and with 64 levels per channel (change both 32s and 32768) |

1. How small can the forest or the MLP get before it falls below the ceiling?
2. How does the ceiling move with the number of bins?  Which number of bins is the fairest, and why?
3. If you had to pick one model for a hospital that will process millions of pixels a day, which
   would you pick from Labs 5–6, and why?